# Task 5: Language Modeling (WikiText-2)
**CENG 467 — NLU&G Take-Home Midterm | Student ID: 300201071**

Models: LSTM Language Model, GPT-2

Metrics: Perplexity + Text Generation

In [1]:
!pip install -q "datasets<3.0.0" datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [2]:
import random, math, numpy as np, torch, torch.nn as nn
from collections import Counter
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import GPT2LMHeadModel, GPT2Tokenizer

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


## 1. Load WikiText-2

In [3]:
ds = load_dataset('wikitext', 'wikitext-2-raw-v1')
print(f'Train: {len(ds["train"])}, Val: {len(ds["validation"])}, Test: {len(ds["test"])}')

# Build vocab
MAX_VOCAB = 30000
counter = Counter()
for ex in ds['train']:
    t = ex['text'].strip()
    if t: counter.update(t.lower().split())
vocab = {'<pad>':0, '<unk>':1, '<eos>':2}
for w,_ in counter.most_common(MAX_VOCAB-3): vocab[w]=len(vocab)
print(f'Vocab: {len(vocab)}')

def tokenize_all(split):
    ids=[]
    for ex in split:
        t=ex['text'].strip()
        if t:
            ids.extend([vocab.get(w,1) for w in t.lower().split()])
            ids.append(vocab['<eos>'])
    return np.array(ids, dtype=np.int64)

train_ids=tokenize_all(ds['train']); val_ids=tokenize_all(ds['validation']); test_ids=tokenize_all(ds['test'])
print(f'Tokens — Train: {len(train_ids):,}, Val: {len(val_ids):,}, Test: {len(test_ids):,}')

BS=128; BPTT=64
def batchify(data, bs):
    nb=len(data)//bs; data=data[:nb*bs]
    return torch.LongTensor(data.reshape(bs,-1))

train_d=batchify(train_ids,BS).to(DEVICE)
val_d=batchify(val_ids,BS).to(DEVICE)
test_d=batchify(test_ids,BS).to(DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Train: 36718, Val: 3760, Test: 4358
Vocab: 30000
Tokens — Train: 2,075,677, Val: 216,347, Test: 244,102


## 2. LSTM Language Model

In [4]:
class LSTMLM(nn.Module):
    def __init__(self, vs, ed=256, hd=512, nl=2, do=0.5):
        super().__init__()
        self.emb=nn.Embedding(vs,ed); self.lstm=nn.LSTM(ed,hd,num_layers=nl,batch_first=True,dropout=do)
        self.drop=nn.Dropout(do); self.fc=nn.Linear(hd,vs); self.hd=hd; self.nl=nl
    def forward(self,x,h=None):
        e=self.drop(self.emb(x)); o,h=self.lstm(e,h); return self.fc(self.drop(o)),h
    def init_h(self,bs):
        return (torch.zeros(self.nl,bs,self.hd).to(DEVICE), torch.zeros(self.nl,bs,self.hd).to(DEVICE))

model=LSTMLM(len(vocab)).to(DEVICE)
opt=torch.optim.Adam(model.parameters(),lr=1e-3)
crit=nn.CrossEntropyLoss()
print(f'LSTM LM params: {sum(p.numel() for p in model.parameters()):,}')

def eval_lm(model, data):
    model.eval(); tl=0; nb=0; h=model.init_h(data.size(0))
    with torch.no_grad():
        for i in range(0, data.size(1)-1, BPTT):
            sl=min(BPTT, data.size(1)-1-i)
            if sl==0: continue
            x=data[:,i:i+sl]; y=data[:,i+1:i+1+sl]
            h=tuple(hh.detach() for hh in h)
            o,h=model(x,h); tl+=crit(o.reshape(-1,o.size(-1)),y.reshape(-1)).item(); nb+=1
    return tl/max(nb,1)

for ep in range(5):
    model.train(); tl=0; nb=0; h=model.init_h(BS)
    for i in range(0, train_d.size(1)-1, BPTT):
        sl=min(BPTT, train_d.size(1)-1-i)
        if sl==0: continue
        x=train_d[:,i:i+sl]; y=train_d[:,i+1:i+1+sl]
        h=tuple(hh.detach() for hh in h)
        opt.zero_grad(); o,h=model(x,h)
        loss=crit(o.reshape(-1,o.size(-1)),y.reshape(-1))
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),0.5); opt.step()
        tl+=loss.item(); nb+=1
    vl=eval_lm(model, val_d)
    print(f'  Ep {ep+1}/5: Train PPL={math.exp(min(tl/nb,20)):.2f} Val PPL={math.exp(min(vl,20)):.2f}')

test_loss = eval_lm(model, test_d)
lstm_ppl = math.exp(min(test_loss, 20))
print(f'\nLSTM Test Perplexity: {lstm_ppl:.2f}')

LSTM LM params: 26,748,208
  Ep 1/5: Train PPL=1252.09 Val PPL=625.30
  Ep 2/5: Train PPL=613.98 Val PPL=400.03
  Ep 3/5: Train PPL=440.53 Val PPL=327.02
  Ep 4/5: Train PPL=360.52 Val PPL=288.56
  Ep 5/5: Train PPL=310.21 Val PPL=261.51

LSTM Test Perplexity: 241.90


In [5]:
# LSTM Text Generation
idx2w = {v:k for k,v in vocab.items()}
def gen_lstm(seed, maxl=80, temp=0.8):
    model.eval(); toks=seed.lower().split()
    ids=[vocab.get(w,1) for w in toks]; h=model.init_h(1); gen=list(toks)
    with torch.no_grad():
        for tid in ids[:-1]:
            _,h=model(torch.LongTensor([[tid]]).to(DEVICE),h)
        cur=ids[-1]
        for _ in range(maxl):
            o,h=model(torch.LongTensor([[cur]]).to(DEVICE),h)
            logits=o[0,-1]/temp; probs=torch.softmax(logits,0)
            nxt=torch.multinomial(probs,1).item()
            if nxt==vocab['<eos>']: break
            gen.append(idx2w.get(nxt,'<unk>')); cur=nxt
    return ' '.join(gen)

print('\nLSTM Generated Samples:')
for s in ['the', 'in the beginning', 'scientists have discovered', 'the united states', 'natural language processing']:
    print(f'  Seed: "{s}"')
    print(f'  → {gen_lstm(s)[:200]}\n')


LSTM Generated Samples:
  Seed: "the"
  → the anti @-@ yard and <unk> of the time 's year . he stated a <unk> in the idea of the <unk> state . the meeting is the early british 's crew at the end of the book , which had returned to a military 

  Seed: "in the beginning"
  → in the beginning of the congo , finding more operated to the eastern old division . a family team had suggested german railway , a day of the house of a number of tanks and only 18 @,@ 000 . for " the

  Seed: "scientists have discovered"
  → scientists have discovered the scout agency as a grandfather . the family was equated to spend a ethics in the week , of the classic , and the same museum of the <unk> nor .

  Seed: "the united states"
  → the united states . the film of the flow of the breeding stood is the spot to power . the film remained from the local song of the church and the storm 's review , and one of the album , is being up t

  Seed: "natural language processing"
  → natural language processing , 

## 3. GPT-2 Language Model

In [6]:
gpt_tok = GPT2Tokenizer.from_pretrained('gpt2')
gpt_model = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)
gpt_model.eval()

# Compute perplexity on WikiText-2 test
text = '\n'.join([ex['text'] for ex in ds['test'] if ex['text'].strip()])
enc = gpt_tok(text, return_tensors='pt')
input_ids = enc['input_ids'][0]
tl=0.0; nt=0; stride=512
with torch.no_grad():
    for i in tqdm(range(0, len(input_ids)-1, stride), desc='GPT-2 PPL'):
        end=min(i+512, len(input_ids))
        chunk=input_ids[i:end].unsqueeze(0).to(DEVICE)
        if chunk.size(1)<2: continue
        out=gpt_model(chunk, labels=chunk)
        tl+=out.loss.item()*(chunk.size(1)-1); nt+=chunk.size(1)-1

gpt_ppl = math.exp(min(tl/max(nt,1), 20))
print(f'\nGPT-2 Test Perplexity: {gpt_ppl:.2f}')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (283287 > 1024). Running this sequence through the model will result in indexing errors


GPT-2 PPL:   0%|          | 0/554 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



GPT-2 Test Perplexity: 36.10


In [7]:
# GPT-2 Text Generation
print('\nGPT-2 Generated Samples:')
for p in ['The', 'In the beginning', 'Scientists have discovered', 'The United States', 'Natural language processing']:
    inp = gpt_tok.encode(p, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = gpt_model.generate(inp, max_length=80, temperature=0.8, top_k=50, top_p=0.95,
                                  do_sample=True, no_repeat_ngram_size=3, pad_token_id=gpt_tok.eos_token_id)
    print(f'  Prompt: "{p}"')
    print(f'  → {gpt_tok.decode(out[0], skip_special_tokens=True)[:200]}\n')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



GPT-2 Generated Samples:
  Prompt: "The"
  → The world needs to take a hard look at how we're going to deal with the Zika virus, and we're starting to do that. We need to find ways to get rid of it," said James Coyle, director of the Center for 

  Prompt: "In the beginning"
  → In the beginning, it seemed that the world was falling apart as the gods had gone about their business. But after seeing that it was all happening in a matter of minutes, they could no longer believe 

  Prompt: "Scientists have discovered"
  → Scientists have discovered that high-frequency magnetic fields are used to generate energy to create a magnetic field that is extremely stable and is stable enough to be seen by the naked eye.

They d

  Prompt: "The United States"
  → The United States of America, a nation of about 800 million people, has not been able to impose its sovereignty over our lives by force. We have lost control of our borders and our national sovereignt

  Prompt: "Natural language processi

## 4. Final Results Summary (for LaTeX Report)

In [8]:
print('\n' + '='*60)
print('  TASK 5 — FINAL RESULTS (Copy to LaTeX)')
print('='*60)
print(f'{"Model":<10} {"Perplexity (↓)":<20}')
print('-'*30)
print(f'{"LSTM":<10} {lstm_ppl:<20.2f}')
print(f'{"GPT-2":<10} {gpt_ppl:<20.2f}')
print('-'*30)
print(f'\nNote: Lower perplexity = better model')


  TASK 5 — FINAL RESULTS (Copy to LaTeX)
Model      Perplexity (↓)      
------------------------------
LSTM       241.90              
GPT-2      36.10               
------------------------------

Note: Lower perplexity = better model
